# ONNX to LiteRT/TFLite Conversion Notebook

This notebook is conversion-only. It does not train models and does not interrupt the baseline training notebook.

Expected input files are `.onnx` exports created by the baseline comparison notebook, usually under:

```text
/content/drive/MyDrive/shrimp_disease_images/baseline_model_comparison/exports
```

For each ONNX model, the notebook attempts:

1. ONNX validation.
2. ONNX -> TensorFlow SavedModel/Keras model via `onnx2tf` CLI.
3. TensorFlow SavedModel -> float32 `.tflite` LiteRT artifact.
4. Optional dynamic-range quantized `.tflite` artifact.
5. A small TFLite load/invoke smoke test.

All conversion results are saved to a CSV report. Failures are recorded per model so one bad conversion does not stop the whole batch.


In [7]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Install conversion dependencies

`onnx2tf` is the main bridge from ONNX to TensorFlow. TensorFlow then produces `.tflite`, which is the LiteRT-compatible artifact. The versions are intentionally pinned loosely enough for Colab, but if a model fails due to version conflict, rerun this cell after a runtime restart.


In [8]:
import sys, subprocess, importlib.util

packages = [
    "onnx",
    "onnxruntime-gpu",
    "onnxslim",
    "sng4onnx>=1.0.1",
    "onnx_graphsurgeon>=0.3.26",
    "onnx2tf>=1.26.3,<1.29.0",
    "tensorflow<=2.19.1",
    "tf-keras<=2.19.0",
    "ai-edge-litert",
    "pandas",
    "numpy",
]

def import_name(pkg):
    base = pkg.split(">=")[0].split("<=")[0].split("<")[0].split("==")[0]
    return {
        "onnx_graphsurgeon": "onnx_graphsurgeon",
        "tf-keras": "tf_keras",
        "onnxruntime-gpu": "onnxruntime",
        "ai-edge-litert": "ai_edge_litert",
    }.get(base, base.replace("-", "_"))

missing = []
for pkg in packages:
    if importlib.util.find_spec(import_name(pkg)) is None:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
    print("Install complete. If TensorFlow was changed, restart runtime before conversion if imports fail.")
else:
    print("All conversion dependencies appear available.")

# Fail early with a clear message if onnx2tf runtime imports are still incomplete.
try:
    import ai_edge_litert
    import onnx2tf
    print("Verified ai_edge_litert and onnx2tf imports.")
except Exception as exc:
    raise RuntimeError(f"Conversion dependency import failed: {type(exc).__name__}: {exc}")


Installing missing packages: ['ai-edge-litert']
Install complete. If TensorFlow was changed, restart runtime before conversion if imports fail.
Verified ai_edge_litert and onnx2tf imports.


## Configure paths

Change `ONNX_INPUT_DIR` if your ONNX files are stored elsewhere. Outputs go to `litert_conversions` so the original export folder remains untouched.


In [9]:
from pathlib import Path
import os, json, time, shutil, subprocess, sys
import numpy as np
import pandas as pd

ONNX_INPUT_DIR = Path('/content/drive/MyDrive/onnx_models')
CONVERSION_ROOT = Path('/content/drive/MyDrive/onnx_models/litert_models')
SAVEDMODEL_DIR = CONVERSION_ROOT / 'saved_models'
TFLITE_DIR = CONVERSION_ROOT / 'tflite'
REPORT_DIR = CONVERSION_ROOT / 'reports'

for d in [SAVEDMODEL_DIR, TFLITE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

onnx_files = sorted(ONNX_INPUT_DIR.glob('*.onnx'))
print(f'Found {len(onnx_files)} ONNX file(s) in {ONNX_INPUT_DIR}')
for p in onnx_files:
    print(' -', p.name, f'({p.stat().st_size / (1024**2):.2f} MB)')

if not onnx_files:
    print('No ONNX files found. Update ONNX_INPUT_DIR before running conversion cells.')


Found 2 ONNX file(s) in /content/drive/MyDrive/onnx_models
 - efficientnet_b0.onnx (15.30 MB)
 - mobilenet_v3_large.onnx (16.04 MB)


## Conversion helpers

The conversion path is intentionally explicit and auditable. If conversion fails, the report captures the exception and the notebook continues to the next model.


In [10]:
import onnx
import tensorflow as tf

IMG_SIZE = 224
REPRESENTATIVE_BATCHES = 20

def safe_name(path: Path) -> str:
    return path.stem.replace(' ', '_').replace('.', '_')

def validate_onnx(onnx_path: Path):
    model = onnx.load(str(onnx_path))
    onnx.checker.check_model(model)
    return True

def run_onnx2tf(onnx_path: Path, output_dir: Path):
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        '-m',
        'onnx2tf',
        '-i', str(onnx_path),
        '-o', str(output_dir),
        '-osd',
    ]
    completed = subprocess.run(cmd, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError(
            'onnx2tf failed\nSTDOUT:\n' + completed.stdout[-4000:] + '\nSTDERR:\n' + completed.stderr[-4000:]
        )
    return output_dir, completed.stdout[-4000:] + completed.stderr[-4000:]

def find_saved_model_dir(output_dir: Path):
    candidates = []
    for root, dirs, files in os.walk(output_dir):
        root_path = Path(root)
        if (root_path / 'saved_model.pb').exists():
            candidates.append(root_path)
    if candidates:
        return candidates[0]
    if (output_dir / 'saved_model.pb').exists():
        return output_dir
    raise FileNotFoundError(f'No saved_model.pb found under {output_dir}')

def convert_saved_model_to_tflite(saved_model_dir: Path, tflite_path: Path, quantize=False):
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    tflite_path.write_bytes(tflite_model)
    return tflite_path

def smoke_test_tflite(tflite_path: Path):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    inputs = interpreter.get_input_details()
    outputs = interpreter.get_output_details()
    input_info = inputs[0]
    input_shape = input_info['shape']
    if any(dim <= 0 for dim in input_shape):
        input_shape = np.array([1, IMG_SIZE, IMG_SIZE, 3], dtype=np.int32)

    sample = np.random.rand(*input_shape).astype(input_info['dtype'])
    interpreter.set_tensor(input_info['index'], sample)
    interpreter.invoke()
    out = interpreter.get_tensor(outputs[0]['index'])
    return {
        'input_shape': tuple(int(x) for x in input_shape),
        'input_dtype': str(input_info['dtype']),
        'output_shape': tuple(int(x) for x in out.shape),
        'output_dtype': str(out.dtype),
    }

def convert_one_onnx(onnx_path: Path):
    name = safe_name(onnx_path)
    model_saved_root = SAVEDMODEL_DIR / name
    float_tflite = TFLITE_DIR / f'{name}_float32.tflite'
    dynamic_tflite = TFLITE_DIR / f'{name}_dynamic_range.tflite'
    start = time.time()
    result = {
        'model': name,
        'onnx_path': str(onnx_path),
        'onnx_size_mb': round(onnx_path.stat().st_size / (1024**2), 2),
        'saved_model_path': '',
        'float32_tflite_path': '',
        'dynamic_tflite_path': '',
        'float32_size_mb': np.nan,
        'dynamic_size_mb': np.nan,
        'smoke_test': '',
        'status': 'failed',
        'error': '',
        'seconds': np.nan,
    }
    try:
        validate_onnx(onnx_path)
        run_onnx2tf(onnx_path, model_saved_root)
        saved_model_dir = find_saved_model_dir(model_saved_root)
        result['saved_model_path'] = str(saved_model_dir)

        convert_saved_model_to_tflite(saved_model_dir, float_tflite, quantize=False)
        result['float32_tflite_path'] = str(float_tflite)
        result['float32_size_mb'] = round(float_tflite.stat().st_size / (1024**2), 2)
        result['smoke_test'] = json.dumps(smoke_test_tflite(float_tflite))

        try:
            convert_saved_model_to_tflite(saved_model_dir, dynamic_tflite, quantize=True)
            result['dynamic_tflite_path'] = str(dynamic_tflite)
            result['dynamic_size_mb'] = round(dynamic_tflite.stat().st_size / (1024**2), 2)
        except Exception as quant_error:
            result['dynamic_tflite_path'] = ''
            result['error'] += f'Dynamic-range quantization failed: {type(quant_error).__name__}: {quant_error}; '

        result['status'] = 'converted'
    except Exception as exc:
        result['error'] += f'{type(exc).__name__}: {exc}'
    finally:
        result['seconds'] = round(time.time() - start, 1)
    return result


## Run conversion batch

This cell converts every `.onnx` file found in `ONNX_INPUT_DIR`. The report is saved after each model, so partial progress is preserved if Colab disconnects.


In [11]:
conversion_results = []
for onnx_path in onnx_files:
    print('\n' + '=' * 80)
    print('Converting:', onnx_path.name)
    print('=' * 80)
    result = convert_one_onnx(onnx_path)
    conversion_results.append(result)
    display(pd.DataFrame([result]))
    pd.DataFrame(conversion_results).to_csv(REPORT_DIR / 'onnx_to_litert_conversion_report_partial.csv', index=False)

report_df = pd.DataFrame(conversion_results)
report_csv = REPORT_DIR / 'onnx_to_litert_conversion_report.csv'
report_json = REPORT_DIR / 'onnx_to_litert_conversion_report.json'
report_df.to_csv(report_csv, index=False)
report_df.to_json(report_json, orient='records', indent=2)
print('\nFinal conversion report:')
display(report_df)
print('Saved:', report_csv)
print('Saved:', report_json)



Converting: efficientnet_b0.onnx


,model,onnx_path,onnx_size_mb,saved_model_path,float32_tflite_path,dynamic_tflite_path,float32_size_mb,dynamic_size_mb,smoke_test,status,error,seconds
0,efficientnet_b0,/content/drive/MyDrive/onnx_models/efficientne...,15.3,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,15.29,4.32,"{""input_shape"": [1, 224, 224, 3], ""input_dtype...",converted,,28.7



Converting: mobilenet_v3_large.onnx


,model,onnx_path,onnx_size_mb,saved_model_path,float32_tflite_path,dynamic_tflite_path,float32_size_mb,dynamic_size_mb,smoke_test,status,error,seconds
0,mobilenet_v3_large,/content/drive/MyDrive/onnx_models/mobilenet_v...,16.04,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,16.04,4.31,"{""input_shape"": [1, 224, 224, 3], ""input_dtype...",converted,,15.5



Final conversion report:


,model,onnx_path,onnx_size_mb,saved_model_path,float32_tflite_path,dynamic_tflite_path,float32_size_mb,dynamic_size_mb,smoke_test,status,error,seconds
0,efficientnet_b0,/content/drive/MyDrive/onnx_models/efficientne...,15.30,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,15.29,4.32,"{""input_shape"": [1, 224, 224, 3], ""input_dtype...",converted,,28.7
1,mobilenet_v3_large,/content/drive/MyDrive/onnx_models/mobilenet_v...,16.04,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,/content/drive/MyDrive/onnx_models/litert_mode...,16.04,4.31,"{""input_shape"": [1, 224, 224, 3], ""input_dtype...",converted,,15.5


Saved: /content/drive/MyDrive/onnx_models/litert_models/reports/onnx_to_litert_conversion_report.csv
Saved: /content/drive/MyDrive/onnx_models/litert_models/reports/onnx_to_litert_conversion_report.json


## Optional single-file conversion

Use this cell if you want to test one ONNX file at a time. Update `SINGLE_ONNX_PATH` and run.


In [12]:
# SINGLE_ONNX_PATH = Path('/content/drive/MyDrive/shrimp_disease_images/baseline_model_comparison/exports/mobilenet_v3_large.onnx')
# result = convert_one_onnx(SINGLE_ONNX_PATH)
# display(pd.DataFrame([result]))
